# PyTorch — Klasifikasi Multi-Label

**Kapan pakai file ini:** satu dokumen boleh punya beberapa label sekaligus. Contoh: berita ekonomi sekaligus politik.

Versi PyTorch dari `cheatsheet-multilabel.ipynb`.
Alur dan CONFIG dibuat semirip mungkin — yang berbeda hanya bagian model: tidak ada
`TfidfVectorizer` dan `.fit()`, melainkan **vocab → tensor → arsitektur → training loop manual**.

**Evaluasinya tetap memakai scikit-learn** (`classification_report`, `confusion_matrix`,
`f1_score`) karena metriknya sama saja, dan §8 menghitung pembanding TF-IDF supaya angkanya jujur.

```
intip file -> ambil kolom -> tokenisasi -> vocab+tensor -> model -> training loop
-> evaluasi (sklearn) -> prediksi -> output
```

Teori: `cheatsheet_pytorch.ipynb` · Semua opsi: `latihan_pytorch.ipynb`

---
## §1 · CONFIG — ubah di sini saja

In [1]:
# ---------------- DATA: struktur file ----------------
PATH   = "data/berita_multilabel.csv"
SEP    = None
HEADER = "infer"
ENC    = None

# ---------------- DATA: kolom ----------------
TEXT_COL   = None
LABEL_COL  = "labels"    # berisi label ganda, mis. "ekonomi|politik"
LABEL_MAP  = None
MULTILABEL = True        # <- wajib True
PEMISAH    = "|"
SAMPLE     = None

BAHASA = "id"

# ---------------- PREPROCESSING (True/False) ----------------
LOWERCASE   = True
MASK        = True      # URL/mention/angka -> token generik
STOPWORD    = False     # untuk neural sering justru DIBIARKAN: urutan kata ikut informatif
JAGA_NEGASI = True

# ---------------- TEKS -> TENSOR ----------------
MIN_FREQ  = 2           # kata dengan frekuensi < ini jadi <unk>
MAX_VOCAB = 20000
MAX_LEN   = 48          # token per dokumen, sisanya dipotong

# ---------------- ARSITEKTUR ----------------
ARSITEKTUR    = "cnn"   # meanpool | cnn | lstm | gru
DIM           = 100     # dimensi embedding          (semua arsitektur)
HIDDEN        = 128     # unit LSTM/GRU              (lstm, gru)
N_FILTER      = 100     # filter per ukuran kernel   (cnn)
KERNEL        = (3, 4, 5)                          # (cnn)
BIDIRECTIONAL = True    # dua arah                   (lstm, gru)
DROPOUT       = 0.3

# ---------------- TRAINING ----------------
EPOCHS       = 30
BATCH        = 64
LR           = 1e-3
WEIGHT_DECAY = 0.0
SEIMBANGKAN  = True

TEST_SIZE, VAL_SIZE, SEED = 0.2, 0.15, 42

CONTOH_BARU = ["pemerintah menaikkan anggaran subsidi energi tahun depan",
               "klub sepak bola itu meraih pendanaan investasi dari luar negeri"]

---
## §2 · Intip struktur file

Jalankan sebelum apa pun. Kalau nama kolom / separator / encoding tidak sesuai dugaan,
perbaiki di CONFIG lalu ulangi sel ini.

In [2]:
import re, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score


def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

# ---- lihat 3 baris pertama file MENTAH ----
print("\n=== isi mentah 3 baris pertama ===")
with open(PATH, encoding="utf-8", errors="replace") as f:
    for i, baris in zip(range(3), f):
        print(f"  {i}: {baris.rstrip()[:110]}")

_sep = SEP or ("\t" if PATH.endswith((".tsv", ".tab")) else ",")
for _enc in ([ENC] if ENC else ["utf-8", "latin-1", "cp1252"]):
    try:
        raw = pd.read_csv(PATH, sep=_sep, header=HEADER, encoding=_enc,
                          engine="python", on_bad_lines="skip")
        break
    except (UnicodeDecodeError, UnicodeError):
        continue

print(f"\n=== terbaca: {raw.shape[0]} baris x {raw.shape[1]} kolom "
      f"(sep={_sep!r}, encoding={_enc}) ===")
print("nama kolom :", list(raw.columns))
print("\njumlah nilai unik per kolom:")
print(raw.nunique().to_string())

_teks = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
_tebak_teks = max(_teks, key=lambda c: raw[c].astype(str).str.len().mean()) if _teks else None
_kand = [(raw[c].nunique(), c) for c in raw.columns
         if c != _tebak_teks and 2 <= raw[c].nunique() <= 200]
print("\n=== saran untuk CONFIG ===")
print(f"TEXT_COL   = {_tebak_teks!r}")
print(f"LABEL_COL  = {min(_kand)[1]!r}" if _kand else "LABEL_COL  = ?")

torch 2.6.0+cu124 | device: cuda | NVIDIA GeForce RTX 3060 Laptop GPU

=== isi mentah 3 baris pertama ===
  0: text,labels
  1: Tim nasional menang tiga gol tanpa balas pada laga kualifikasi,olahraga
  2: Pelatih menyebut kondisi pemain membaik menjelang laga final,olahraga

=== terbaca: 88 baris x 2 kolom (sep=',', encoding=utf-8) ===
nama kolom : ['text', 'labels']

jumlah nilai unik per kolom:
text      88
labels     9

=== saran untuk CONFIG ===
TEXT_COL   = 'text'
LABEL_COL  = 'labels'


---
## §3 · Ambil kolom teks & label

In [3]:
text_col, label_col = TEXT_COL, LABEL_COL
if text_col is None:
    kand = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
    text_col = max(kand, key=lambda c: raw[c].astype(str).str.len().mean())
if label_col is None:
    batas = 200 if MULTILABEL else 20
    kand = [(raw[c].nunique(), c) for c in raw.columns
            if c != text_col and 2 <= raw[c].nunique() <= batas]
    label_col = min(kand)[1]
print(f"kolom teks: {text_col!r} | kolom label: {label_col!r}")

df = raw.rename(columns={text_col: "text", label_col: "label"})[["text", "label"]].copy()
df["text"] = df["text"].astype(str).str.strip()
if not MULTILABEL:
    df["label"] = df["label"].apply(lambda v: v.strip().lower() if isinstance(v, str) else v)
if LABEL_MAP:
    df["label"] = df["label"].map(LABEL_MAP).fillna(df["label"])

n0 = len(df)
df = df[df["text"].str.len() >= 3]
df = df[~df["text"].str.lower().isin({"nan", "none", "na", "-"})]
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
if SAMPLE and SAMPLE < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE, random_state=SEED,
                             stratify=None if MULTILABEL else df["label"])
    df = df.reset_index(drop=True)
print(f"setelah dibersihkan: {n0} -> {len(df)} baris")

kolom teks: 'text' | kolom label: 'labels'
setelah dibersihkan: 88 -> 88 baris


In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

df["label_list"] = df["label"].map(lambda v: [x.strip() for x in str(v).split(PEMISAH) if x.strip()])
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["label_list"]).astype("float32")
KELAS = list(mlb.classes_)
N_OUT = len(KELAS)

print("label unik:", KELAS)
print("rata-rata label per dokumen:", round(Y.sum(axis=1).mean(), 2))
print()
print(pd.DataFrame(Y[:5].astype(int), columns=KELAS).to_string(index=False))

label unik: ['ekonomi', 'olahraga', 'politik', 'teknologi']
rata-rata label per dokumen: 1.12

 ekonomi  olahraga  politik  teknologi
       0         1        0          0
       0         1        0          0
       0         1        0          1
       0         1        0          0
       1         1        0          0


---
## §4 · Tokenisasi

Untuk jalur neural, teks tidak diubah jadi TF-IDF melainkan jadi **daftar token**, lalu token
diganti indeks angka di §5. Stopword sering justru **tidak** dibuang untuk neural, karena urutan
dan kata fungsi ikut membawa informasi bagi CNN/LSTM.

In [5]:
STOP_EN = {"i","me","my","we","you","your","he","she","it","they","them","this","that","is","are",
           "was","were","be","the","a","an","and","but","or","of","at","by","for","with","to",
           "from","in","on","so","than","too","very","just","now","have","has","had","do","did"}
STOP_ID = {"yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada","adalah","ada","saya",
           "kamu","dia","kami","kita","mereka","akan","sudah","juga","atau","karena","saja"}
NEGASI  = {"no","not","never","nor","cannot"} | {"tidak","bukan","tanpa","jangan","belum","kurang"}

stop = (STOP_EN if BAHASA == "en" else STOP_ID)
if JAGA_NEGASI:
    stop = stop - NEGASI


def tokenize(teks):
    t = str(teks)
    if LOWERCASE:
        t = t.lower()
    if MASK:
        t = re.sub(r"http\S+|www\.\S+|\b\S+\.(?:com|org|net|ly|id|co)\S*", " urltoken ", t)
        t = re.sub(r"@\w+", " usertoken ", t)
        t = re.sub(r"\b\d+\b", " numtoken ", t)
    kata = re.findall(r"[a-zA-Z]+", t)
    if STOPWORD:
        kata = [w for w in kata if w not in stop]
    return kata or ["kosongtoken"]


print("contoh token:", tokenize(df["text"].iloc[0])[:12])

contoh token: ['tim', 'nasional', 'menang', 'tiga', 'gol', 'tanpa', 'balas', 'pada', 'laga', 'kualifikasi']


---
## §5 · Vocab → tensor → DataLoader

Empat langkah yang menggantikan satu baris `TfidfVectorizer`: bangun **vocab** (hanya dari data
latih), ubah token jadi **indeks**, **padding** supaya satu batch sama panjang, lalu bungkus di
`DataLoader`. Panjang asli tiap kalimat disimpan karena dipakai LSTM dan mean pooling.

In [6]:
from collections import Counter

PAD, UNK = 0, 1

# --- split dulu, baru bangun vocab: vocab HANYA boleh dari data latih ---
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED,
                                  stratify=None if MULTILABEL else df["label"])
idx_tr, idx_val = train_test_split(idx_tr, test_size=VAL_SIZE, random_state=SEED,
                                   stratify=None if MULTILABEL else df["label"].iloc[idx_tr])
print("train:", len(idx_tr), "| val:", len(idx_val), "| test:", len(idx_te))

c = Counter(w for t in df["text"].iloc[idx_tr] for w in tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, n in c.most_common(MAX_VOCAB) if n >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
print("ukuran vocab:", len(itos))


def encode(teks):
    return [stoi.get(w, UNK) for w in tokenize(teks)][:MAX_LEN] or [UNK]


class DatasetTeks(Dataset):
    def __init__(self, idx):
        self.ids = [encode(t) for t in df["text"].iloc[idx]]
        self.y = Y[idx]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return torch.tensor(self.ids[i], dtype=torch.long), self.y[i]


MIN_LEN = max(KERNEL) if ARSITEKTUR == "cnn" else 1


def collate(batch):
    urut, label = zip(*batch)
    panjang = torch.tensor([len(x) for x in urut], dtype=torch.long)
    padded = pad_sequence(urut, batch_first=True, padding_value=PAD)
    if padded.size(1) < MIN_LEN:
        padded = F.pad(padded, (0, MIN_LEN - padded.size(1)), value=PAD)
    y = torch.tensor(np.array(label), dtype=torch.float if MULTILABEL else torch.long)
    return padded, panjang, y


mk = lambda idx, sh: DataLoader(DatasetTeks(idx), batch_size=BATCH, shuffle=sh, collate_fn=collate)
dl_tr, dl_val, dl_te = mk(idx_tr, True), mk(idx_val, False), mk(idx_te, False)

xb, lb, yb = next(iter(dl_tr))
print("bentuk batch:", tuple(xb.shape), "-> (batch, panjang terpanjang di batch)")

train: 59 | val: 11 | test: 18
ukuran vocab: 92
bentuk batch: (59, 11) -> (batch, panjang terpanjang di batch)


---
## §6 · Model — pilih arsitektur lewat `ARSITEKTUR` di CONFIG

In [7]:
class MeanPool(nn.Module):
    def __init__(self, n_vocab, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, DIM, padding_idx=PAD)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(DIM, n_out)

    def forward(self, x, panjang):
        e = self.emb(x)
        mask = (x != PAD).unsqueeze(-1).float()
        return self.fc(self.drop((e * mask).sum(1) / mask.sum(1).clamp(min=1)))


class CNNTeks(nn.Module):                      # Kim 2014, slide 02b hal. 10
    def __init__(self, n_vocab, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, DIM, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(DIM, N_FILTER, k) for k in KERNEL])
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(N_FILTER * len(KERNEL), n_out)

    def forward(self, x, panjang=None):
        e = self.emb(x).transpose(1, 2)
        f = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]
        return self.fc(self.drop(torch.cat(f, dim=1)))


class RNNTeks(nn.Module):                      # LSTM/GRU, slide 02b hal. 11
    def __init__(self, n_vocab, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, DIM, padding_idx=PAD)
        kelas = nn.LSTM if ARSITEKTUR == "lstm" else nn.GRU
        self.rnn = kelas(DIM, HIDDEN, batch_first=True, bidirectional=BIDIRECTIONAL)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN * (2 if BIDIRECTIONAL else 1), n_out)

    def forward(self, x, panjang):
        packed = pack_padded_sequence(self.emb(x), panjang.cpu(), batch_first=True,
                                      enforce_sorted=False)
        keluar = self.rnn(packed)[1]
        h = keluar[0] if isinstance(keluar, tuple) else keluar
        h = torch.cat([h[-2], h[-1]], dim=1) if BIDIRECTIONAL else h[-1]
        return self.fc(self.drop(h))


set_seed()
model = {"meanpool": MeanPool, "cnn": CNNTeks,
         "lstm": RNNTeks, "gru": RNNTeks}[ARSITEKTUR](len(itos), N_OUT).to(device)
print(f"arsitektur : {ARSITEKTUR}")
print(f"parameter  : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

arsitektur : cnn
parameter  : 130,704


---
## §7 · Loss + training loop

Lima baris di dalam loop, urutannya tidak boleh salah:
`zero_grad → forward → loss → backward → step`. Bobot terbaik disimpan menurut **skor validasi**,
bukan epoch terakhir — kalau tidak, yang kamu evaluasi adalah model yang sudah mulai overfit.

In [8]:
# Multilabel: tiap label diputuskan sendiri-sendiri -> loss biner per label.
# pos_weight adalah padanan class_weight untuk multilabel: satu bobot per label,
# dihitung dari (jumlah negatif / jumlah positif) label itu di data latih.
# Tanpa ini model cenderung menjawab "bukan" untuk semua label -> prediksi kosong.
if SEIMBANGKAN:
    freq = Y[idx_tr].sum(axis=0)
    pw = torch.tensor((len(idx_tr) - freq) / np.maximum(freq, 1),
                      dtype=torch.float, device=device)
    print("frekuensi tiap label:", dict(zip(KELAS, freq.astype(int).tolist())))
    print("pos_weight          :", dict(zip(KELAS, pw.cpu().numpy().round(2).tolist())))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
else:
    criterion = nn.BCEWithLogitsLoss()
    print("loss: BCEWithLogitsLoss tanpa pos_weight")

frekuensi tiap label: {'ekonomi': 18, 'olahraga': 17, 'politik': 16, 'teknologi': 17}
pos_weight          : {'ekonomi': 2.2799999713897705, 'olahraga': 2.4700000286102295, 'politik': 2.690000057220459, 'teknologi': 2.4700000286102295}


In [9]:
@torch.no_grad()
def prediksi(loader):
    model.eval()
    P, T = [], []
    for x, panjang, y in loader:
        logits = model(x.to(device), panjang.to(device))
        P.append((torch.sigmoid(logits) > 0.5).int().cpu() if MULTILABEL
                 else logits.argmax(1).cpu())
        T.append(y.cpu())
    return torch.cat(P).numpy(), torch.cat(T).numpy()


optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
rata = "micro" if MULTILABEL else "macro"

terbaik, state = -1, None
for ep in range(1, EPOCHS + 1):
    model.train()
    total, t0 = 0.0, time.time()
    for x, panjang, y in dl_tr:
        x, panjang, y = x.to(device), panjang.to(device), y.to(device)
        optimizer.zero_grad()                       # 1. hapus gradien batch sebelumnya
        loss = criterion(model(x, panjang), y)      # 2. forward + hitung loss
        loss.backward()                             # 3. hitung gradien
        optimizer.step()                            # 4. perbarui bobot
        total += loss.item() * y.size(0)
    pv, tv = prediksi(dl_val)
    f1v = f1_score(tv, pv, average=rata, zero_division=0)
    if f1v > terbaik:                               # simpan bobot terbaik menurut VALIDASI
        terbaik, state = f1v, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"  epoch {ep:2d}  loss {total/len(dl_tr.dataset):.4f}  val_f1 {f1v:.4f}"
          f"  ({time.time()-t0:.1f}s)")

model.load_state_dict(state)
print(f"\nbobot terbaik dikembalikan (val_f1 {terbaik:.4f})")

  epoch  1  loss 0.9847  val_f1 0.3478  (0.4s)
  epoch  2  loss 0.9149  val_f1 0.3571  (0.0s)
  epoch  3  loss 0.8416  val_f1 0.3333  (0.0s)
  epoch  4  loss 0.7572  val_f1 0.3158  (0.0s)
  epoch  5  loss 0.7158  val_f1 0.4211  (0.0s)
  epoch  6  loss 0.6408  val_f1 0.4444  (0.0s)
  epoch  7  loss 0.5786  val_f1 0.4444  (0.0s)
  epoch  8  loss 0.5323  val_f1 0.4444  (0.0s)
  epoch  9  loss 0.5221  val_f1 0.4444  (0.0s)
  epoch 10  loss 0.4593  val_f1 0.4444  (0.0s)
  epoch 11  loss 0.4222  val_f1 0.3529  (0.0s)
  epoch 12  loss 0.3992  val_f1 0.3158  (0.0s)
  epoch 13  loss 0.3658  val_f1 0.3158  (0.0s)
  epoch 14  loss 0.3278  val_f1 0.3158  (0.0s)
  epoch 15  loss 0.2897  val_f1 0.4000  (0.0s)
  epoch 16  loss 0.2706  val_f1 0.4000  (0.0s)
  epoch 17  loss 0.2504  val_f1 0.3810  (0.0s)
  epoch 18  loss 0.2335  val_f1 0.3810  (0.0s)
  epoch 19  loss 0.2160  val_f1 0.3810  (0.0s)
  epoch 20  loss 0.2020  val_f1 0.3810  (0.0s)
  epoch 21  loss 0.1777  val_f1 0.3810  (0.0s)


  epoch 22  loss 0.1586  val_f1 0.3810  (0.0s)
  epoch 23  loss 0.1548  val_f1 0.3810  (0.0s)
  epoch 24  loss 0.1436  val_f1 0.4545  (0.0s)
  epoch 25  loss 0.1323  val_f1 0.3810  (0.0s)
  epoch 26  loss 0.1192  val_f1 0.3810  (0.0s)
  epoch 27  loss 0.1015  val_f1 0.4000  (0.0s)
  epoch 28  loss 0.1052  val_f1 0.4762  (0.0s)
  epoch 29  loss 0.0998  val_f1 0.4762  (0.0s)
  epoch 30  loss 0.0921  val_f1 0.4762  (0.0s)

bobot terbaik dikembalikan (val_f1 0.4762)


---
## §8 · Evaluasi (pakai scikit-learn) + pembanding TF-IDF

In [10]:
pred, ytrue = prediksi(dl_te)

print("f1 micro:", round(f1_score(ytrue, pred, average="micro", zero_division=0), 3))
print("f1 macro:", round(f1_score(ytrue, pred, average="macro", zero_division=0), 3), "\n")
print(classification_report(ytrue, pred, target_names=KELAS, zero_division=0))
print("subset accuracy (semua label tepat):", round(float((pred == ytrue).all(axis=1).mean()), 3))
print("dokumen tanpa satu pun label      :", f"{(pred.sum(axis=1) == 0).mean():.0%}")

# ---- pembanding: TF-IDF + OneVsRest LogisticRegression ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

i_tr = np.concatenate([idx_tr, idx_val])
klasik = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
                   ("clf", OneVsRestClassifier(
                       LogisticRegression(max_iter=1000, class_weight="balanced")))])
klasik.fit(df["text"].iloc[i_tr], Y[i_tr])
f1_klasik = f1_score(ytrue, klasik.predict(df["text"].iloc[idx_te]),
                     average="micro", zero_division=0)

print(f"\nPyTorch ({ARSITEKTUR:9s}) f1_micro = "
      f"{f1_score(ytrue, pred, average='micro', zero_division=0):.3f}")
print(f"TF-IDF + OvR LogReg  f1_micro = {f1_klasik:.3f}")

f1 micro: 0.4
f1 macro: 0.4 

              precision    recall  f1-score   support

     ekonomi       0.67      0.50      0.57         4
    olahraga       1.00      0.20      0.33         5
     politik       1.00      0.14      0.25         7
   teknologi       0.33      0.67      0.44         3

   micro avg       0.55      0.32      0.40        19
   macro avg       0.75      0.38      0.40        19
weighted avg       0.82      0.32      0.37        19
 samples avg       0.33      0.33      0.33        19

subset accuracy (semua label tepat): 0.333
dokumen tanpa satu pun label      : 39%

PyTorch (cnn      ) f1_micro = 0.400
TF-IDF + OvR LogReg  f1_micro = 0.533


---
## §9 · Prediksi teks baru

In [11]:
@torch.no_grad()
def prediksi_teks(daftar):
    model.eval()
    ids = [torch.tensor(encode(t)) for t in daftar]
    panjang = torch.tensor([len(i) for i in ids])
    x = pad_sequence(ids, batch_first=True, padding_value=PAD)
    if x.size(1) < MIN_LEN:
        x = F.pad(x, (0, MIN_LEN - x.size(1)), value=PAD)
    prob = torch.sigmoid(model(x.to(device), panjang.to(device))).cpu().numpy()
    return [(t, [KELAS[j] for j in np.where(p > 0.5)[0]], p.round(2)) for t, p in zip(daftar, prob)]


for teks, label, skor in prediksi_teks(CONTOH_BARU):
    print(f"  {label}  <- {teks[:55]}")
    print(f"     skor tiap label: {dict(zip(KELAS, skor.tolist()))}")

  ['ekonomi', 'politik']  <- pemerintah menaikkan anggaran subsidi energi tahun depa
     skor tiap label: {'ekonomi': 0.8100000023841858, 'olahraga': 0.019999999552965164, 'politik': 0.6600000262260437, 'teknologi': 0.07000000029802322}
  []  <- klub sepak bola itu meraih pendanaan investasi dari lua
     skor tiap label: {'ekonomi': 0.25, 'olahraga': 0.20999999344348907, 'politik': 0.05999999865889549, 'teknologi': 0.41999998688697815}


---
## §10 · Output — simpan model & tulis hasil

Untuk PyTorch yang disimpan adalah **`state_dict`**, bukan objek modelnya. Vocab (`itos`) dan
daftar kelas ikut disimpan — tanpa itu model tidak bisa dipakai lagi karena tidak tahu kata mana
berindeks berapa.

In [12]:
import json

NAMA = PATH.split("/")[-1].split(".")[0]
torch.save({"state_dict": model.state_dict(), "itos": itos, "kelas": KELAS,
            "cfg": dict(ARSITEKTUR=ARSITEKTUR, DIM=DIM, HIDDEN=HIDDEN, N_FILTER=N_FILTER,
                        KERNEL=KERNEL, BIDIRECTIONAL=BIDIRECTIONAL, DROPOUT=DROPOUT,
                        MAX_LEN=MAX_LEN)},
           f"model_pytorch_{NAMA}.pt")

ke_teks = lambda baris: PEMISAH.join([KELAS[j] for j in np.where(baris)[0]])
hasil = pd.DataFrame({"text": df["text"].iloc[idx_te].values,
                      "aktual":   [ke_teks(r) for r in ytrue],
                      "prediksi": [ke_teks(r) for r in pred]})
hasil["tepat_semua"] = hasil["aktual"] == hasil["prediksi"]
hasil.to_csv(f"hasil_pytorch_{NAMA}.csv", index=False)

ringkas = {"dataset": PATH, "case": "multilabel", "arsitektur": ARSITEKTUR,
           "label": KELAS, "vocab": len(itos), "epochs": EPOCHS,
           "n_train": len(idx_tr), "n_test": len(idx_te),
           "f1_micro": round(float(f1_score(ytrue, pred, average="micro", zero_division=0)), 4),
           "f1_macro": round(float(f1_score(ytrue, pred, average="macro", zero_division=0)), 4),
           "subset_accuracy": round(float((pred == ytrue).all(axis=1).mean()), 4)}
with open(f"ringkasan_pytorch_{NAMA}.json", "w") as f:
    json.dump(ringkas, f, indent=2)

print(f"tersimpan: model_pytorch_{NAMA}.pt | hasil_pytorch_{NAMA}.csv"
      f" | ringkasan_pytorch_{NAMA}.json")
print()
print(json.dumps(ringkas, indent=2))
print()
print(hasil.head(4).to_string(index=False))

tersimpan: model_pytorch_berita_multilabel.pt | hasil_pytorch_berita_multilabel.csv | ringkasan_pytorch_berita_multilabel.json

{
  "dataset": "data/berita_multilabel.csv",
  "case": "multilabel",
  "arsitektur": "cnn",
  "label": [
    "ekonomi",
    "olahraga",
    "politik",
    "teknologi"
  ],
  "vocab": 92,
  "epochs": 30,
  "n_train": 59,
  "n_test": 18,
  "f1_micro": 0.4,
  "f1_macro": 0.3998,
  "subset_accuracy": 0.3333
}

                                                                            text    aktual  prediksi  tepat_semua
        Presiden mengumumkan perombakan kabinet setelah evaluasi kinerja menteri   politik                  False
                  Tim nasional menang tiga gol tanpa balas pada laga kualifikasi  olahraga  olahraga         True
    Pembaruan perangkat lunak menambal celah keamanan yang dilaporkan bulan lalu teknologi teknologi         True
Perusahaan teknologi merilis ponsel pintar terbaru dengan kamera resolusi tinggi teknologi teknologi        

---
## Catatan khusus case multi-label (PyTorch)

Tiga hal yang berbeda dari case lain — dan ketiganya di bagian **loss dan keputusan**, bukan
di arsitektur:

| Hal | Single-label | Multi-label |
|---|---|---|
| Target `Y` | vektor indeks kelas (`long`) | matriks 0/1 (`float`) |
| Loss | `CrossEntropyLoss` | **`BCEWithLogitsLoss`** |
| Keputusan | `logits.argmax(1)` | **`sigmoid(logits) > 0.5`** |
| Metrik | f1 macro | f1 micro **dan** macro |

**Kenapa sigmoid, bukan softmax?** Softmax memaksa jumlah probabilitas semua kelas = 1, artinya
menaikkan satu kelas otomatis menurunkan yang lain — itu asumsi "hanya boleh satu label".
Sigmoid menghitung tiap label sendiri-sendiri, jadi sebuah dokumen bisa mendapat skor tinggi
untuk dua label sekaligus.

### `pos_weight` — padanan `class_weight` untuk multilabel

Sama seperti di versi sklearn, tiap sub-masalah biner ("ekonomi vs bukan ekonomi") otomatis
timpang, sehingga model belajar bahwa menjawab "bukan" hampir selalu aman. Di PyTorch obatnya
adalah `pos_weight` pada `BCEWithLogitsLoss` — satu bobot per label, dihitung dari
(jumlah negatif / jumlah positif) label itu di data latih.

Hasil uji pada dataset ini (61 dokumen latih):

| Setelan | f1 micro | prediksi kosong |
|---|---|---|
| CNN tanpa `pos_weight`, 30 epoch | 0.167 | 72% |
| CNN tanpa `pos_weight`, 60 epoch | 0.357 | 50% |
| **CNN + `pos_weight`, 30 epoch** | **0.400** | **39%** |
| TF-IDF + OvR LogReg (sklearn) | **0.533** | — |

Dua hal yang jujur harus dicatat: `pos_weight` jelas membantu, **tapi model klasik tetap
menang telak**. Dengan 61 dokumen latih, neural network tidak punya cukup data untuk
mempelajari representasi kata dari nol. Ini contoh bagus untuk laporan.

- Prediksi masih bisa **kosong**. §8 mencetak persentasenya. Kalau mengganggu, turunkan ambang
  atau paksa minimal satu label: `pred[kosong, prob[kosong].argmax(1)] = 1`.
- Ambang 0,5 bukan angka suci — menurunkannya menaikkan recall, menurunkan precision.